In [ ]:
import pandas as pd
import src.config
from datasets import get_dataset_config_names

repo = "IntelLabs/AI-Peer-Review-Detection-Benchmark"
configs = get_dataset_config_names(repo)
print(configs[:20], "…", len(configs))


In [ ]:
from huggingface_hub import list_repo_files

repo_id = "IntelLabs/AI-Peer-Review-Detection-Benchmark"
files = list_repo_files(repo_id=repo_id, repo_type="dataset")
csvs = [f for f in files if f.endswith(".csv")]
print("CSV:", len(csvs))
print("\n".join(csvs[:20]))

In [ ]:
from datasets import load_dataset

data_file = "calibration/gpt4o/ICLR2022.calibration.gpt-4o.csv"

ds = load_dataset(
    "csv",
    data_files={"data": f"hf://datasets/{repo_id}/{data_file}"},
    split="data",
)
print(ds)
print(ds.column_names)
print(ds[0])

In [ ]:
from datasets import load_dataset

repo_id = "IntelLabs/AI-Peer-Review-Detection-Benchmark"
file_path = "test/gpt4o/ICLR2022.test.gpt-4o.csv"  # exemple

ds = load_dataset(
    "csv",
    data_files={"data": f"hf://datasets/{repo_id}/{file_path}"},
    split="data",
)

print(ds.column_names)
# check distribution
labels = ds["label"]
print({0: labels.count(0), 1: labels.count(1)})


In [ ]:
ds = load_dataset("csv",
                  data_files="hf://datasets/IntelLabs/AI-Peer-Review-Detection-Benchmark/test/gpt4o/ICLR2017.test.gpt-4o.csv",
                  split="train")

print(ds.column_names)

# Affiche quelques exemples par label
print("Label 0 example:", ds.filter(lambda x: x["label"] == 0)[0])
print("Label 1 example:", ds.filter(lambda x: x["label"] == 1)[0])


In [ ]:
ds_small = ds.shuffle(seed=42)
df = ds_small.to_pandas()

In [ ]:
df

In [ ]:
df[df["label"] == 0]

In [ ]:
list_repo_files(repo_id=repo_id, repo_type="dataset")

# Experimentation with dataset structure


In [ ]:
 from src.data.dataset_aprdb import AprdbDataset
dataset = AprdbDataset(config={}, subset="test", keep_meta=True)

In [ ]:
type(dataset.data)

In [ ]:
len(dataset)

In [ ]:
print(len(dataset.data))
df = dataset.data.data.to_pandas()
print(df)

In [ ]:
set_data = set(df['content'])
print(len(set_data))

In [ ]:
import re
from collections import Counter
import numpy as np


def basic_audit(dataset, max_items=None):
    labels = []
    lengths = []
    flags = Counter()
    texts = []
    datas_for_label = {
        0: {
            "empty": 0,
            "url": 0,
            "bullet": 0,
            "multi_newlines": 0,
            "bracket_refs": 0,
            "length_content": []
        },
        1: {
            "empty": 0,
            "url": 0,
            "bullet": 0,
            "multi_newlines": 0,
            "bracket_refs": 0,
            "length_content": []
        }
    }

    url_re = re.compile(r"https?://|www\.")
    bullet_re = re.compile(r"(^|\n)\s*([-•*]|\d+\)|\d+\.)\s+")
    multi_nl_re = re.compile(r"\n{3,}")
    bracket_ref_re = re.compile(r"\[\d+\]")

    n = len(dataset) if max_items is None else min(len(dataset), max_items)

    for i in range(n):
        item = dataset[i]
        y = int(item["label"])
        x = item["content"] or ""
        x = x.strip()

        datas_for_label[y]["empty"] += int(len(x) == 0)
        datas_for_label[y]["url"] += int(bool(url_re.search(x)))
        datas_for_label[y]["bullet"] += int(bool(bullet_re.search(x)))
        datas_for_label[y]["multi_newlines"] += int(bool(multi_nl_re.search(x)))
        datas_for_label[y]["bracket_refs"] += int(bool(bracket_ref_re.search(x)))
        datas_for_label[y]["length_content"].append(len(x.strip()))


        labels.append(y)
        texts.append(x)
        lengths.append(len(x.split()))

        flags["empty"] += int(len(x) == 0)
        flags["url"] += int(bool(url_re.search(x)))
        flags["bullets"] += int(bool(bullet_re.search(x)))
        flags["multi_newlines"] += int(bool(multi_nl_re.search(x)))
        flags["bracket_refs"] += int(bool(bracket_ref_re.search(x)))

    label_counts = Counter(labels)

    # exact duplicates
    dup_count = n - len(set(texts))

    def summarize(arr):
        arr = np.array(arr)
        return {
            "p": [float(np.percentile(arr, i)) for i in range(0, 99)],
            "p10": float(np.percentile(arr, 10)),
            "p50": float(np.percentile(arr, 50)),
            "p90": float(np.percentile(arr, 90)),
            "mean": float(arr.mean()),
            "std": float(arr.std()),
        }

    # lengths by class
    lengths_by_class = {}
    for cls in sorted(label_counts):
        cls_lengths = [l for l, y in zip(lengths, labels) if y == cls]
        lengths_by_class[cls] = summarize(cls_lengths)

    return {
        "N": n,
        "label_counts": dict(label_counts),
        "flags_for_label": datas_for_label,
        "lengths_by_class_words": lengths_by_class,
        "flags": dict(flags),
        "exact_duplicate_count": dup_count,
    }


In [ ]:
basic_a = basic_audit(dataset)

In [ ]:
basic_a

In [ ]:
basic_a['flags_for_label'][0]['bullet']

In [ ]:

basic_a['flags_for_label'][1]['bullet']


In [ ]:
res = basic_a

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x0 = np.array(basic_a["flags_for_label"][0]["length_content"], dtype=float)
x1 = np.array(basic_a["flags_for_label"][1]["length_content"], dtype=float)

# def display_histogram(x0, x1):
#     # Bins communs (important pour comparer)
#     xmin = min(x0.min(), x1.min())
#     xmax = max(x0.max(), x1.max())
#     bins = np.linspace(xmin, xmax, int(np.sqrt(len(x0))))  # ajuste 60 si besoin
#
#     plt.figure()
#     plt.hist(x0, bins=bins, alpha=0.5, label="label 0")
#     plt.hist(x1, bins=bins, alpha=0.5, label="label 1")
#     plt.xlabel("Length (words)")
#     plt.ylabel("Count")
#     plt.title("Histogrammes superposés des longueurs")
#     plt.legend()
#     plt.grid(True, alpha=0.3)
#     plt.show()

In [ ]:
from src.data_analyse.aprb_analyse.aprb_utils import *

plot_data_x0 = PlotData(x0, "Human Written Text")
plot_data_x1 = PlotData(x1, "Machin Generated Text")

display_histogram([plot_data_x1, plot_data_x0])

In [ ]:

# display_histogram(x0, x1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Suppose que tu as un dict `res` du style:
# res["lengths_by_class_words"][0]["p"] et res["lengths_by_class_words"][1]["p"]

p0_t = np.array(res["lengths_by_class_words"][0]["p"], dtype=float)
p1_t = np.array(res["lengths_by_class_words"][1]["p"], dtype=float)

def display_percentile(p0, p1):
    # Axe des percentiles (ici 0..98 si tu fais range(0,99))
    x = np.arange(len(p0))

    plt.figure()
    plt.plot(x, p0, label="label 0 (humain)")
    plt.plot(x, p1, label="label 1 (généré)")
    plt.xlabel("Percentile")
    plt.ylabel("Longueur (mot/token)")
    plt.title("Courbes de percentiles des longueurs (0 vs 1)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
from transformers import AutoTokenizer
import re
from collections import Counter
import numpy as np

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def basic_audit_tokenized(dataset, max_items=None, tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")):
    labels = []
    lengths = []
    flags = Counter()
    texts = []
    datas_for_label = {
        0: {
            "empty": 0,
            "url": 0,
            "bullet": 0,
            "multi_newlines": 0,
            "bracket_refs": 0,
            "length_content": []
        },
        1: {
            "empty": 0,
            "url": 0,
            "bullet": 0,
            "multi_newlines": 0,
            "bracket_refs": 0,
            "length_content": []
        }
    }

    url_re = re.compile(r"https?://|www\.")
    bullet_re = re.compile(r"(^|\n)\s*([-•*]|\d+\)|\d+\.)\s+")
    multi_nl_re = re.compile(r"\n{3,}")
    bracket_ref_re = re.compile(r"\[\d+\]")

    n = len(dataset) if max_items is None else min(len(dataset), max_items)

    for i in range(n):
        item = dataset[i]
        y = int(item["label"])
        x = item["content"] or ""
        x = x.strip()

        datas_for_label[y]["empty"] += int(len(x) == 0)
        datas_for_label[y]["url"] += int(bool(url_re.search(x)))
        datas_for_label[y]["bullet"] += int(bool(bullet_re.search(x)))
        datas_for_label[y]["multi_newlines"] += int(bool(multi_nl_re.search(x)))
        datas_for_label[y]["bracket_refs"] += int(bool(bracket_ref_re.search(x)))

        # Tokenizing the content
        tokens = tokenizer.encode(x, add_special_tokens=False)
        datas_for_label[y]["length_content"].append(len(tokens))

        labels.append(y)
        texts.append(x)
        lengths.append(len(tokens))

        flags["empty"] += int(len(x) == 0)
        flags["url"] += int(bool(url_re.search(x)))
        flags["bullets"] += int(bool(bullet_re.search(x)))
        flags["multi_newlines"] += int(bool(multi_nl_re.search(x)))
        flags["bracket_refs"] += int(bool(bracket_ref_re.search(x)))

    label_counts = Counter(labels)

    # exact duplicates
    dup_count = n - len(set(texts))

    def summarize(arr):
        arr = np.array(arr)
        return {
            "p": [float(np.percentile(arr, i)) for i in range(0, 99)],
            "p10": float(np.percentile(arr, 10)),
            "p50": float(np.percentile(arr, 50)),
            "p90": float(np.percentile(arr, 90)),
            "mean": float(arr.mean()),
            "std": float(arr.std()),
        }

    # lengths by class
    lengths_by_class = {}
    for cls in sorted(label_counts):
        cls_lengths = [l for l, y in zip(lengths, labels) if y == cls]
        lengths_by_class[cls] = summarize(cls_lengths)

    return {
        "N": n,
        "label_counts": dict(label_counts),
        "flags_for_label": datas_for_label,
        "lengths_by_class_tokens": lengths_by_class,
        "flags": dict(flags),
        "exact_duplicate_count": dup_count,
    }

basic_a_tokenized = basic_audit_tokenized(dataset)


In [ ]:
basic_a_tokenized

In [ ]:
p0 = np.array(basic_a_tokenized["lengths_by_class_tokens"][0]["p"], dtype=float)
p1 = np.array(basic_a_tokenized["lengths_by_class_tokens"][1]["p"], dtype=float)
x0 = np.array(basic_a_tokenized["flags_for_label"][0]["length_content"], dtype=float)
x1 = np.array(basic_a_tokenized["flags_for_label"][1]["length_content"], dtype=float)


display_histogram(x0, x1)
display_percentile(p0, p1)

## Looking for duplicate

In [ ]:
# import re, hashlib, unicodedata
# from collections import defaultdict, Counter
#
# _ws_re = re.compile(r"\s+")
#
# def normalize_text(s: str, do_lower: bool = False) -> str:
#     if s is None:
#         s = ""
#     s = unicodedata.normalize("NFKC", s)
#     s = s.strip()
#     s = _ws_re.sub(" ", s)  # collapse whitespace
#     if do_lower:
#         s = s.lower()
#     return s
#
# def text_hash(s: str) -> str:
#     # sha1 suffisant ici
#     return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()
#
# def find_duplicate_groups(dataset, normalized=True, do_lower=False, max_items=None):
#     groups = defaultdict(list)   # hash -> [idx...]
#     reps = {}                    # hash -> representative text (pour preview)
#
#     n = len(dataset) if max_items is None else min(len(dataset), max_items)
#
#     for i in range(n):
#         item = dataset[i]
#         x = item["content"] if isinstance(item, dict) else item[1]  # adapte si besoin
#         x = x if x is not None else ""
#
#         key_text = normalize_text(x, do_lower=do_lower) if normalized else x
#         h = text_hash(key_text)
#
#         groups[h].append(i)
#         # garde un représentant (version originale) pour afficher
#         if h not in reps:
#             reps[h] = x
#
#     dup_hashes = [h for h, idxs in groups.items() if len(idxs) > 1]
#     return groups, reps, dup_hashes


In [ ]:
# def summarize_duplicates(dataset, groups, dup_hashes):
#     sizes = [len(groups[h]) for h in dup_hashes]
#     size_counter = Counter(sizes)
#
#     # check inter-label
#     inter_label = []
#     for h in dup_hashes:
#         labels = []
#         for idx in groups[h]:
#             item = dataset[idx]
#             y = int(item["label"]) if isinstance(item, dict) else int(item[0])
#             labels.append(y)
#         if len(set(labels)) > 1:
#             inter_label.append(h)
#
#     return {
#         "n_duplicate_groups": len(dup_hashes),
#         "n_items_in_duplicate_groups": sum(sizes),
#         "max_group_size": max(sizes) if sizes else 0,
#         "group_size_hist": dict(size_counter),
#         "n_inter_label_groups": len(inter_label),
#         "inter_label_hashes": inter_label[:20],  # échantillon
#     }


In [ ]:
# grp, reps, gh = find_duplicate_groups(dataset)

In [ ]:
# summarize_duplicates(dataset, grp, gh)

In [ ]:
# def top_k_groups(groups, dup_hashes, k=10):
#     return sorted(dup_hashes, key=lambda h: len(groups[h]), reverse=True)[:k]

In [ ]:
# top_k_groups(grp, gh)

In [ ]:
# def show_group(dataset, groups, h, max_chars=800):
#     idxs = groups[h]
#     print(f"Group hash={h} | size={len(idxs)}")
#
#     labels = []
#     for k, idx in enumerate(idxs):
#         item = dataset[idx]
#         y = int(item["label"]) if isinstance(item, dict) else int(item[0])
#         x = item["content"] if isinstance(item, dict) else item[1]
#         labels.append(y)
#         print(f"\n--- item {k} | idx={idx} | label={y} | len_chars={len(x)} ---")
#         print(x[:max_chars] + (" ..." if len(x) > max_chars else ""))
#
#     print(f"\nLabels in group: {sorted(set(labels))} (counts={Counter(labels)})")

In [ ]:
# show_group(dataset, grp, "25efed448a13bf9f2f1318230d131c1a8935fdf0", max_chars = 10000)

In [ ]:
# grp

In [ ]:
# reps

In [ ]:
# gh

In [ ]:
ex = "This is a strong submission regarding one of the most important and recently introduced methods in neural networks - generative adversarial networks. The authors analyze theoretically the convergence of GANs and discuss the stability of GANs. Both are very important. To the best of my knowledge, this is one of the first theoretical papers about GANs and the paper, contrary to most of the submissions in the field, actually provides deep theoretical insight into this architecture. The stability issues regarding GANs are extremely important since the first proposed versions of GANs architecture were very unstable and did not work well in practice. Theorems 2.4-2.6 are novel and introduces mathematical techniques are interesting. I have some technical questions regarding the proof of Theorem 2.5 but these are pretty minor."
print(df[df['content'] == ex])

In [ ]:
from datasets import load_dataset
from huggingface_hub import list_repo_files

# print(list_repo_files("IntelLabs/AI-Peer-Review-Detection-Benchmark", repo_type="dataset"))
ds_test_c_icrl2017 = load_dataset(
    "csv",
    data_files={
        "data": f"hf://datasets/IntelLabs/AI-Peer-Review-Detection-Benchmark/test/claude/ICLR2017.test.claude-sonnet-3.5v2.csv"},
    split="data",
)

In [ ]:
ds_test_c_icrl2017

In [ ]:
df2 = ds_test_c_icrl2017.to_pandas()

In [ ]:
print(df2.iloc[1587])
print(df2.iloc[1933])

In [ ]:
print(df2[df2['review'] == ex])